# Clustering Analysis of Colorado Bird Sightings

This notebook uses clustering analysis on `birdsong_df` to answer the following research questions:

9. How has the prevalence of different species of birds changed over time?
10. What are the typical flight patterns of different bird species?

The notebook also captures dataset snapshots before and after each major transformation, documents model assumptions and tuning choices, and evaluates clustering quality with Silhouette Score and Davies-Bouldin Index.

## Why K-Means Clustering Was Chosen

`birdsong_df` is largely unlabelled for the abovementioned questions. There is no ground-truth label that says which species belong to the same temporal prevalence profile or which species share similar flight behavior. That makes unsupervised learning the right modeling family.

K-Means is used here because:

- The transformed feature sets are numeric and can be standardized.
- We want interpretable centroids that summarize each cluster.
- The dataset is large enough that a fast centroid-based method is practical.
- We can tune the number of clusters using internal validation metrics.

## Model Assumptions

K-Means assumes:

- Clusters are reasonably compact and separable in feature space.
- Euclidean distance is meaningful after feature scaling.
- Features with larger raw units should not dominate, so scaling is required.
- The chosen number of clusters `k` is not known in advance and must be tuned.

For the flight-pattern analysis, an additional practical assumption is made: because the dataset contains observations rather than true tracked trajectories, flight patterns are approximated using seasonal and monthly geographic movement signatures rather than literal path reconstruction.

In [67]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
sns.set_theme(style='whitegrid', context='talk')

In [68]:
# Use the in-memory DataFrame when it already exists; otherwise fall back to the saved CSV.
if 'birdsong_df' not in globals():
    birdsong_df = pd.read_csv('..\\data\\birdsong.csv')

birdsong_df = birdsong_df.copy()
birdsong_df.head()

,common_name,date,bird_count,county,family,total_monthly_precipitation,monthly_max_temp,monthly_min_temp,urban_population_level,county_total_population,is_fire,is_flood,bird_count_adjusted_for_population,season,year,month,log_bird_count
0,Band-tailed Pigeon,2021-01-01,1.0000,Garfield,Pigeons and Doves,23.0158,1.4122,-10.4589,4,"62,178.4800",0,0,0.0000,Winter,2021,1,0.6931
1,Pine Warbler,2021-01-01,1.0000,Boulder,New World Warblers,9.8737,3.8300,-8.2778,5,"333,404.0640",0,0,0.0000,Winter,2021,1,0.6931
2,White-winged Scoter,2021-01-01,1.0000,Larimer,"Ducks, Geese, and Waterfowl",10.8259,1.8541,-10.4572,5,"361,938.5280",0,0,0.0000,Winter,2021,1,0.6931
3,Brown Thrasher,2021-01-01,1.0000,Boulder,Mockingbirds and Thrashers,9.8737,3.8300,-8.2778,5,"333,404.0640",0,0,0.0000,Winter,2021,1,0.6931
4,Bonaparte's Gull,2021-01-01,2.0000,Pueblo,"Gulls, Terns, and Skimmers",14.1278,8.5667,-8.4000,5,"169,507.2960",0,0,0.0000,Winter,2021,1,1.0986


## Shared Utilities

Hyperparameter tuning is handled by testing multiple values of `k` and selecting the value that maximizes Silhouette Score, with Davies-Bouldin Index used as a secondary quality check.

In [69]:
# Test over multiple values of k to find the one that gives the best evaluation score
def evaluate_kmeans_grid(X, k_values=range(2, 9), random_state=42):
    rows = []
    for k in k_values:
        model = KMeans(n_clusters=k, random_state=random_state, n_init=20)
        labels = model.fit_predict(X)
        rows.append({
            'k': k,
            'silhouette_score': silhouette_score(X, labels),
            'davies_bouldin_index': davies_bouldin_score(X, labels)
        })
    scores = pd.DataFrame(rows)
    best_k = scores.sort_values(['silhouette_score', 'davies_bouldin_index'], ascending=[False, True]).iloc[0]['k']
    return scores, int(best_k)

# Create a model with the best k value and fit the data to it
def fit_final_kmeans(X, best_k, random_state=42):
    model = KMeans(n_clusters=best_k, random_state=random_state, n_init=20)
    labels = model.fit_predict(X)  # cluster labels are numbers 1, 2, ..., k
    metrics = {
        'silhouette_score': silhouette_score(X, labels),
        'davies_bouldin_index': davies_bouldin_score(X, labels)
    }
    return model, labels, metrics

## 9. Species Prevalence Changes Over Time

To study how species prevalence changes over time, each species is represented by a monthly prevalence profile. Prevalence is defined as total observed count per month, and a small `log1p` transform is applied to reduce the influence of unusually large counts.

In [70]:
species_monthly = (
    birdsong_df
    .groupby(['common_name', 'date'], as_index=False)
    .agg(monthly_bird_count=('bird_count', 'sum'))
)
species_monthly['log_monthly_bird_count'] = np.log1p(species_monthly['monthly_bird_count'])

### Snapshot Before Prevalence Pivot

In [71]:
species_monthly.head(10)

,common_name,date,monthly_bird_count,log_monthly_bird_count
0,Acadian Flycatcher,2023-05-01,1.0000,0.6931
1,Acadian Flycatcher,2024-05-01,1.0000,0.6931
2,Acadian Flycatcher,2024-06-01,1.0000,0.6931
3,Acorn Woodpecker,2021-01-01,9.0000,2.3026
4,Acorn Woodpecker,2021-02-01,6.0000,1.9459
5,Acorn Woodpecker,2021-03-01,2.0000,1.0986
6,Acorn Woodpecker,2021-05-01,3.0000,1.3863
7,Acorn Woodpecker,2021-06-01,16.0000,2.8332
8,Acorn Woodpecker,2021-07-01,15.0000,2.7726
9,Acorn Woodpecker,2021-08-01,18.0000,2.9444


### Data Transformation Steps

Reshape the species prevalence data into a format that can be clustered.

- `species_prevalence_wide` pivots the monthly prevalence table so that each row represents one bird species and each column represents one month in the time series. The values are the log-transformed monthly bird counts. This turns each species into a time-series feature vector that captures how its prevalence changes over time.
- `.fillna(0)` replaces missing species-month combinations with `0`, meaning the species had no recorded prevalence in that month. This is necessary because clustering requires a complete numeric matrix.
- `species_prevalence_scaled` applies `StandardScaler()` to the pivoted matrix so the monthly columns are on a comparable scale. In the clustering analysis, this prevents months with larger raw count ranges from dominating the distance calculations and helps the model group species by the shape of their prevalence pattern rather than just absolute magnitude.

Together, these transformations produce a clean species-by-time matrix that is suitable for K-Means clustering of prevalence trends.

In [72]:
species_prevalence_wide = (
    species_monthly
    .pivot(index='common_name', columns='date', values='log_monthly_bird_count')
    .fillna(0)
)

species_prevalence_scaled = pd.DataFrame(
    StandardScaler().fit_transform(species_prevalence_wide),
    index=species_prevalence_wide.index,
    columns=species_prevalence_wide.columns
)

### Snapshot After Prevalence Scaling

In [73]:
species_prevalence_scaled

date,2021-01-01,2021-02-01,2021-03-01,2021-04-01,2021-05-01,2021-06-01,2021-07-01,2021-08-01,2021-09-01,2021-10-01,2021-11-01,2021-12-01,2022-01-01,2022-02-01,2022-03-01,2022-04-01,2022-05-01,2022-06-01,2022-07-01,2022-08-01,2022-09-01,2022-10-01,2022-11-01,2022-12-01,2023-01-01,2023-02-01,2023-03-01,2023-04-01,2023-05-01,2023-06-01,2023-07-01,2023-08-01,2023-09-01,2023-10-01,2023-11-01,2023-12-01,2024-01-01,2024-02-01,2024-03-01,2024-04-01,2024-05-01,2024-06-01,2024-07-01,2024-08-01,2024-09-01,2024-10-01,2024-11-01,2024-12-01,2025-01-01,2025-02-01,2025-03-01,2025-04-01,2025-05-01,2025-06-01,2025-07-01,2025-08-01,2025-09-01,2025-10-01,2025-11-01,2025-12-01
common_name,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Acadian Flycatcher,-0.7252,-0.7075,-0.7781,-0.9609,-1.1959,-0.9999,-0.9614,-0.9789,-1.0026,-0.8916,-0.7521,-0.7275,-0.7370,-0.6958,-0.7828,-1.0130,-1.2310,-1.0429,-0.9977,-0.9930,-1.0591,-0.9477,-0.7880,-0.6998,-0.7074,-0.6906,-0.7673,-1.0294,-0.8686,-1.0482,-1.0083,-1.0052,-1.0558,-0.9267,-0.7939,-0.7645,-0.7451,-0.7096,-0.7980,-1.0365,-0.8670,-0.6736,-1.0163,-1.0318,-1.0759,-0.9639,-0.8012,-0.7493,-0.7223,-0.7247,-0.8037,-0.9978,-1.0983,-1.0488,-1.0365,-1.0538,-1.0458,-0.9577,-0.7972,-0.7383
Acorn Woodpecker,0.3430,0.2323,-0.2563,-0.9609,-0.4769,0.4286,0.3719,0.4025,-0.1486,0.1784,-0.7521,-0.7275,0.7433,0.3795,-0.7828,-1.0130,-0.1491,-0.6853,-0.0644,0.1517,0.0815,-0.2756,-0.2471,-0.6998,0.1238,0.3825,-0.7673,-1.0294,-0.3850,0.2016,0.0051,-0.0934,-0.5253,0.0993,-0.7939,-0.7645,-0.7451,0.2446,-0.7980,-0.1350,-0.3857,0.6834,0.6050,-0.3727,0.0821,-0.1665,-0.8012,0.2211,-0.7223,-0.1862,-0.4590,-0.2774,-0.3647,0.0140,0.4177,0.1111,0.0263,0.2203,0.3261,-0.4090
African Collared-Dove,-0.7252,-0.7075,-0.7781,-0.9609,-1.1959,-0.6504,-0.9614,-0.9789,-0.6722,-0.8916,-0.2255,-0.7275,-0.7370,-0.6958,-0.7828,-1.0130,-1.2310,-1.0429,-0.9977,-0.9930,-1.0591,-0.9477,-0.7880,-0.6998,-0.1978,-0.6906,-0.2433,-0.4869,-0.8686,-1.0482,-1.0083,-0.6804,-1.0558,-0.9267,-0.1294,-0.7645,-0.7451,-0.7096,-0.7980,-1.0365,-1.2311,-1.0300,-1.0163,-1.0318,-0.7412,-0.9639,-0.8012,-0.7493,-0.7223,-0.7247,-0.8037,-0.9978,-1.0983,-0.4488,-0.4939,-1.0538,-1.0458,-0.9577,-0.7972,-0.7383
Alder Flycatcher,-0.7252,-0.7075,-0.7781,-0.9609,-0.0017,-0.9999,-0.9614,-0.4635,-1.0026,-0.8916,-0.7521,-0.7275,-0.7370,-0.6958,-0.7828,-1.0130,-0.0330,-0.1186,-0.9977,-0.6737,-0.7294,-0.9477,-0.7880,-0.6998,-0.7074,-0.6906,-0.7673,-1.0294,-0.1369,-1.0482,-1.0083,-1.0052,-0.7211,-0.9267,-0.7939,-0.7645,-0.7451,-0.7096,-0.7980,-1.0365,-0.6540,-1.0300,-1.0163,-1.0318,-1.0759,-0.9639,-0.8012,-0.7493,-0.7223,-0.7247,-0.8037,-0.9978,-0.3647,-1.0488,-1.0365,-0.7171,-0.7076,-0.9577,-0.7972,-0.7383
American Avocet,-0.7252,-0.7075,0.8372,1.6491,1.2740,1.7376,1.6701,1.6764,1.7315,1.9665,1.3300,-0.7275,-0.7370,-0.6958,1.0712,1.7761,1.3326,1.4224,1.3891,1.5488,1.4058,2.2145,0.9336,-0.6998,-0.7074,-0.6906,0.9002,1.6729,1.5486,1.5198,2.2471,1.2646,1.6029,1.7456,0.9978,-0.7645,-0.7451,0.6183,1.2162,1.5475,1.4281,1.5499,1.9519,1.5388,2.1345,1.4080,0.7228,-0.7493,-0.7223,-0.0452,1.0050,1.3464,1.4865,1.7092,1.3971,1.2719,1.5394,1.6502,0.9085,-0.7383
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Yellow-rumped x Townsend's Warbler (hybrid),-0.7252,-0.7075,-0.7781,-0.9609,-1.1959,-0.9999,-0.9614,-0.9789,-1.0026,-0.8916,-0.7521,-0.7275,-0.7370,-0.6958,-0.7828,-1.0130,-1.2310,-1.0429,-0.9977,-0.9930,-1.0591,-0.9477,-0.7880,-0.6998,-0.7074,-0.6906,-0.7673,-1.0294,-1.2344,-1.0482,-1.0083,-1.0052,-0.7211,-0.9267,-0.7939,-0.7645,-0.7451,-0.7096,-0.7980,-1.0365,-1.2311,-1.0300,-1.0163,-1.0318,-1.0759,-0.9639,-0.8012,-0.7493,-0.7223,-0.7247,-0.8037,-0.9978,-1.0983,-1.0488,-1.0365,-1.0538,-1.0458,-0.9577,-0.7972,-0.7383
Yellow-throated Vireo,-0.7252,-0.7075,-0.7781,-0.9609,0.2086,-0.6504,-0.9614

### Applying K-Means Clustering Models

K-means clustering is done for values of `k` from 2 to 9. For each value of `k`, the Silhouette Score and Davies Bouldin Index are computed.

In [74]:
prevalence_scores, prevalence_best_k = evaluate_kmeans_grid(species_prevalence_scaled, k_values=range(2, 9))
prevalence_model, prevalence_labels, prevalence_metrics = fit_final_kmeans(species_prevalence_scaled, prevalence_best_k)

prevalence_scores

,k,silhouette_score,davies_bouldin_index
0,2,0.4960,0.7972
1,3,0.4808,0.8617
2,4,0.4566,0.9542
3,5,0.4583,0.9241
4,6,0.4310,1.0360
5,7,0.4067,1.1874
6,8,0.3941,1.2088


The model that uses the `k` value with the best scores is used to generate the clusters.  
Then, each observation is assigned its corresponding cluster label.  
Finally, the following are computed for each cluster:
1. Number of species in the cluster
2. Average standardized prevalence of the species in the cluster
3. Average time trend for the species in the cluster: Generally increasing, decreasing, or stable over time

In [75]:
species_prevalence_clusters = species_prevalence_scaled.copy()
species_prevalence_clusters['cluster'] = prevalence_labels
species_prevalence_clusters['avg_scaled_prevalence'] = species_prevalence_scaled.mean(axis=1)
species_prevalence_clusters['prevalence_trend'] = (
    species_prevalence_wide.iloc[:, -12:].mean(axis=1) - species_prevalence_wide.iloc[:, :12].mean(axis=1)
)

prevalence_cluster_summary = (
    species_prevalence_clusters
    .groupby('cluster')
    .agg(
        species_count=('avg_scaled_prevalence', 'size'),
        mean_scaled_prevalence=('avg_scaled_prevalence', 'mean'),
        mean_trend=('prevalence_trend', 'mean')
    )
    .sort_values('mean_trend', ascending=False)
)

prevalence_cluster_summary

,species_count,mean_scaled_prevalence,mean_trend
cluster,,,
0,356,-0.5710,0.0094
1,211,0.9633,-0.1001


### Prevalence Cluster Interpretation

- **Cluster 0:** This cluster contains more species overall and has a slightly positive average `mean_trend`, which suggests a broadly stable-to-mildly increasing prevalence profile over the period from 2021 to 2025. Its negative `mean_scaled_prevalence` indicates that many species in this group are relatively lower-prevalence species compared with the dataset average, even if some are becoming more common later on. Example species include **Cassia Crossbill**, **Yellow-billed Loon**, **Northern Parula**, **Muscovy Duck**, and **Anhinga**.
- **Cluster 1:** This cluster has a higher average `mean_scaled_prevalence`, so the species here tend to be relatively more prevalent overall, but the negative `mean_trend` suggests that, on average, this group declines over the period from 2021 to 2025. In other words, these species are comparatively common in the dataset but show weaker prevalence later on. Example species include **Gambel's Quail**, **Greater Roadrunner**, **Sagebrush Sparrow**, **American Barn Owl**, and **Golden-crowned Kinglet**.

These interpretations should be read together with `prevalence_cluster_summary`: higher `mean_scaled_prevalence` reflects relatively stronger overall prevalence, while `mean_trend` shows whether the cluster tends to increase or decrease over time.

## 10. Typical Flight Patterns by Species

This analysis estimates flight-pattern types using the county locations recorded in `birdsong_df` across time. Each species is represented by its county-level distribution over months. That lets the model group species with similar geographic spread, county turnover, and seasonal concentration patterns. To turn those clusters into interpretable movement corridors, county centroids are estimated from the raw eBird latitude and longitude observations in `ebird_co.csv`, then aggregated into month-to-month cluster paths.

In [76]:
species_monthly_county = (
    birdsong_df
    .groupby(['common_name', 'month', 'county'], as_index=False)
    .agg(monthly_count=('bird_count', 'sum'))
)

species_monthly_county['log_monthly_count'] = np.log1p(species_monthly_county['monthly_count'])

### Snapshot Before Flight-Pattern Feature Engineering

In [77]:
species_monthly_county.head(10)

,common_name,month,county,monthly_count,log_monthly_count
0,Acadian Flycatcher,5,Lincoln,1.0000,0.6931
1,Acadian Flycatcher,5,Pueblo,1.0000,0.6931
2,Acadian Flycatcher,6,Pueblo,1.0000,0.6931
3,Acorn Woodpecker,1,La Plata,37.0000,3.6376
4,Acorn Woodpecker,2,La Plata,30.0000,3.4340
5,Acorn Woodpecker,3,La Plata,3.0000,1.3863
6,Acorn Woodpecker,4,La Plata,8.0000,2.1972
7,Acorn Woodpecker,5,La Plata,18.0000,2.9444
8,Acorn Woodpecker,5,Larimer,1.0000,0.6931
9,Acorn Woodpecker,5,Montezuma,1.0000,0.6931


### Feature Engineering Steps

This block converts county-by-month bird observations into features that can be clustered into typical flight-pattern types.

- `species_county_month_wide` creates a wide matrix where each row is a species and each column is a specific month-county combination such as `m03_Boulder`. The values are log-transformed bird counts. In the clustering analysis, this captures where a species tends to appear and when it appears there.
- `monthly_species_summary` summarizes each species by month. It keeps track of total monthly count and the number of active counties. In the clustering analysis, this helps describe how broadly a species is distributed in a typical month.
- `county_turnover` converts each species-month into a set of counties. That makes it possible to compare one month's county footprint with the next month's footprint.
- The `turnover_rows` loop calculates `avg_county_turnover`, which measures how much a species changes counties from month to month. Values closer to `0` mean the species tends to stay in the same counties, while larger values indicate stronger geographic shifting over time.
- `flight_pattern_features` combines several summary traits for each species: how many counties it visits, how many months it is active, its average count, its average monthly county spread, and its month-to-month county turnover. These are the compact behavioral features used to describe movement style.
- `log_total_observed_count` is added to reduce the effect of extremely large raw counts. This keeps very common species from dominating the clustering simply because of scale rather than movement behavior.

Together, these transformations give the clustering model both a detailed month-county distribution matrix and a smaller set of summary movement features, so clusters reflect movement behavior rather than just raw abundance.

In [78]:
species_county_month_wide = (
    species_monthly_county
    .assign(month_county=lambda df: 'm' + df['month'].astype(str).str.zfill(2) + '_' + df['county'].str.replace(' ', '_', regex=False))
    .pivot(index='common_name', columns='month_county', values='log_monthly_count')
    .fillna(0)
)

monthly_species_summary = (
    species_monthly_county
    .groupby(['common_name', 'month'], as_index=False)
    .agg(
        monthly_total_count=('monthly_count', 'sum'),
        counties_active=('county', 'nunique')
    )
)

county_turnover = (
    species_monthly_county
    .groupby(['common_name', 'month'])['county']
    .apply(lambda x: set(x))
    .reset_index(name='county_set')
    .sort_values(['common_name', 'month'])
)

turnover_rows = []
for species_name, species_df in county_turnover.groupby('common_name'):
    sets = list(species_df['county_set'])
    if len(sets) <= 1:
        turnover_rows.append({'common_name': species_name, 'avg_county_turnover': 0.0})
        continue

    turnovers = []
    for prev_set, curr_set in zip(sets[:-1], sets[1:]):
        union_size = len(prev_set | curr_set)
        if union_size == 0:
            turnovers.append(0.0)
        else:
            turnovers.append(1 - (len(prev_set & curr_set) / union_size))
    turnover_rows.append({'common_name': species_name, 'avg_county_turnover': float(np.mean(turnovers))})

county_turnover_features = pd.DataFrame(turnover_rows)

flight_pattern_features = (
    species_monthly_county
    .groupby('common_name', as_index=False)
    .agg(
        counties_visited=('county', 'nunique'),
        active_months=('month', 'nunique'),
        avg_monthly_count=('monthly_count', 'mean'),
        total_observed_count=('monthly_count', 'sum')
    )
    
    .merge(
        monthly_species_summary.groupby('common_name', as_index=False).agg(avg_active_counties=('counties_active', 'mean')),
        on='common_name',
        how='left'
    )

    .merge(county_turnover_features, on='common_name', how='left')
)

flight_pattern_features['log_total_observed_count'] = np.log1p(flight_pattern_features['total_observed_count'])
flight_pattern_features = flight_pattern_features.drop(columns='total_observed_count')

Next, select summary movement features and standardize their values by applying `StandardScaler()` to transform them onto comparable scale to prevent larger values from dominating the distance calculations.  
Additionally, standardize the values in the matrix of species and the month-county combinations.  
Then, join them together.

In [79]:
flight_numeric_cols = [
    'counties_visited', 'active_months', 'avg_monthly_count',
    'avg_active_counties', 'avg_county_turnover', 'log_total_observed_count'
]

flight_summary_scaled = pd.DataFrame(
    StandardScaler().fit_transform(flight_pattern_features.set_index('common_name')[flight_numeric_cols]),
    index=flight_pattern_features['common_name'],
    columns=flight_numeric_cols
)

flight_distribution_scaled = pd.DataFrame(
    StandardScaler().fit_transform(species_county_month_wide),
    index=species_county_month_wide.index,
    columns=species_county_month_wide.columns
)

flight_pattern_scaled = flight_summary_scaled.join(flight_distribution_scaled, how='inner')

### Snapshot After Flight-Pattern Feature Engineering

In [80]:
flight_pattern_scaled.head(10)

,counties_visited,active_months,avg_monthly_count,avg_active_counties,avg_county_turnover,log_total_observed_count,m01_Adams,m01_Alamosa,m01_Arapahoe,m01_Archuleta,m01_Baca,m01_Bent,m01_Boulder,m01_Broomfield,m01_Chaffee,m01_Cheyenne,m01_Clear_Creek,m01_Costilla,m01_Crowley,m01_Custer,m01_Delta,m01_Denver,m01_Dolores,m01_Douglas,m01_Eagle,m01_El_Paso,m01_Elbert,m01_Fremont,m01_Garfield,m01_Gilpin,m01_Grand,m01_Gunnison,m01_Hinsdale,m01_Huerfano,m01_Jackson,m01_Jefferson,m01_Kiowa,m01_Kit_Carson,m01_La_Plata,m01_Lake,m01_Larimer,m01_Las_Animas,m01_Lincoln,m01_Logan,m01_Mesa,m01_Mineral,m01_Moffat,m01_Montezuma,m01_Montrose,m01_Morgan,...,m12_Crowley,m12_Custer,m12_Delta,m12_Denver,m12_Dolores,m12_Douglas,m12_Eagle,m12_El_Paso,m12_Elbert,m12_Fremont,m12_Garfield,m12_Gilpin,m12_Grand,m12_Gunnison,m12_Huerfano,m12_Jackson,m12_Jefferson,m12_Kiowa,m12_Kit_Carson,m12_La_Plata,m12_Lake,m12_Larimer,m12_Las_Animas,m12_Lincoln,m12_Logan,m12_Mesa,m12_Mineral,m12_Moffat,m12_Montezuma,m12_Montrose,m12_Morgan,m12_Otero,m12_Ouray,m12_Park,m12_Phillips,m12_Pitkin,m12_Prowers,m12_Pueblo,m12_Rio_Blanco,m12_Rio_Grande,m12_Routt,m12_Saguache,m12_San_Juan,m12_San_Miguel,m12_Sedgwick,m12_Summit,m12_Teller,m12_Washington,m12_Weld,m12_Yuma
common_name,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Acadian Flycatcher,-1.2135,-1.4986,-0.2947,-1.0281,-0.1262,-1.5475,-0.2730,-0.1417,-0.5276,-0.2276,-0.2608,-0.2985,-0.5823,-0.3462,-0.2687,-0.0772,-0.1706,-0.1036,-0.1240,-0.1211,-0.2756,-0.4249,-0.2216,-0.3479,-0.2464,-0.5774,-0.1192,-0.3995,-0.2990,-0.1164,-0.2207,-0.1260,-0.0691,-0.1730,-0.1137,-0.5721,-0.2099,-0.1283,-0.4074,-0.0970,-0.5755,-0.1629,-0.1369,-0.1993,-0.5222,-0.1029,-0.1661,-0.4465,-0.3602,-0.1311,...,-0.1491,-0.1732,-0.2318,-0.4210,-0.1419,-0.3768,-0.2051,-0.5508,-0.1002,-0.3988,-0.2309,-0.1241,-0.1988,-0.1835,-0.2593,-0.0923,-0.6030,-0.1369,-0.1794,-0.3964,-0.0968,-0.5806,-0.2061,-0.1489,-0.2701,-0.4847,-0.0967,-0.1329,-0.4497,-0.3122,-0.2467,-0.3264,-0.2622,-0.2058,-0.1221,-0.3109,-0.2314,-0.5593,-0.1247,-0.1175,-0.1765,-0.0952,-0.0528,-0.1571,-0.1168,-0.2384,-0.1943,-0.1476,-0.4563,-0.2288
Acorn Woodpecker,-1.1150,1.0386,-0.0438,-1.0192,-1.1223,0.0526,-0.2730,-0.1417,-0.5276,-0.2276,-0.2608,-0.2985,-0.5823,-0.3462,-0.2687,-0.0772,-0.1706,-0.1036,-0.1240,-0.1211,-0.2756,-0.4249,-0.2216,-0.3479,-0.2464,-0.5774,-0.1192,-0.3995,-0.2990,-0.1164,-0.2207,-0.1260,-0.0691,-0.1730,-0.1137,-0.5721,-0.2099,-0.1283,2.7306,-0.0970,-0.5755,-0.1629,-0.1369,-0.1993,-0.5222,-0.1029,-0.1661,-0.4465,-0.3602,-0.1311,...,-0.1491,-0.1732,-0.2318,-0.4210,-0.1419,-0.3768,-0.2051,-0.5508,-0.1002,-0.3988,-0.2309,-0.1241,-0.1988,-0.1835,-0.2593,-0.0923,-0.6030,-0.1369,-0.1794,1.6732,-0.0968,-0.5806,-0.2061,-0.1489,-0.2701,-0.4847,-0.0967,-0.1329,-0.4497,-0.3122,-0.2467,-0.3264,-0.2622,-0.2058,-0.1221,-0.3109,-0.2314,-0.5593,-0.1247,-0.1175,-0.1765,-0.0952,-0.0528,-0.1571,-0.1168,-0.2384,-0.1943,-0.1476,-0.4563,-0.2288
African Collared-Dove,-1.0164,0.2774,-0.2855,-1.0459,1.5173,-0.9600,-0.2730,-0.1417,-0.5276,-0.2276,-0.2608,-0.2985,0.0388,-0.3462,-0.2687,-0.0772,-0.1706,-0.1036,-0.1240,-0.1211,-0.2756,-0.4249,-0.2216,-0.3479,-0.2464,-0.5774,-0.1192,-0.3995,-0.2990,-0.1164,-0.2207,-0.1260,-0.0691,-0.1730,-0.1137,-0.5721,-0.2099,-0.1283,-0.4074,-0.0970,-0.5755,-0.1629,-0.1369,-0.1993,-0.5222,-0.1029,-0.1661,-0.4465,-0.3602,-0.1311,...,-0.1491,-0.1732,-0.2318,-0.4210,-0.1419,-0.3768,-0.2051,-0.5508,-0.1002,-0.3988,-0.2309,-0.1241,-0.1988,-0.1835,-0.2593,-0.0923,-0.6030,-0.1369,-0.1794,-0.3964,-0.0968,-0.5806,-0.2061,-0.1489,-0.2701,-0.4847,-0.0967,-0.1329,-0.4497,-0.3122,-0.2467,-0.3264,-0.2622,-0.2058,-0.1221,-0.3109,-0.2314,-0.5593,-0.1247,-0.1175,-0.1765,-0.0952,-0.0528,-0.1571,-0.1168,-0.2384,-0.1943,-0.1476,-0.4563,-0.2288
Alder Flycatcher,-0.5235,-0.9912,-0.2839,-0.5743,0.8370,-0.7144,-0.2730,-0.1417,-0.5276,-0.2276,-0.2608,-0.2985,-0.5823,-0.3462,-0.2687,-0.0772,-0.1706,-0.1036,-0.1240,-0.1211,-0.2756,-0.42

### Applying K-Means Clustering Models

K-means clustering is done for values of `k` from 2 to 9. For each value of `k`, the Silhouette Score and Davies Bouldin Index are computed.

In [81]:
flight_scores, flight_best_k = evaluate_kmeans_grid(flight_pattern_scaled[flight_numeric_cols], k_values=range(2, 9))
flight_model, flight_labels, flight_metrics = fit_final_kmeans(flight_pattern_scaled[flight_numeric_cols], flight_best_k)

flight_pattern_scaled['cluster'] = flight_labels
flight_scores

,k,silhouette_score,davies_bouldin_index
0,2,0.4508,0.9134
1,3,0.4816,0.7172
2,4,0.5054,0.6172
3,5,0.4406,0.7538
4,6,0.3818,0.9250
5,7,0.3642,0.9253
6,8,0.3645,0.9337


The model that uses the `k` value with the best scores is used to generate the clusters.  
Then, each observation is assigned its corresponding cluster label.

In [82]:
flight_cluster_summary = (
    flight_pattern_features
    .assign(cluster=flight_labels)
    .groupby('cluster')
    .agg(
        species_count=('common_name', 'size'),
        counties_visited=('counties_visited', 'mean'),
        active_months=('active_months', 'mean'),
        avg_active_counties=('avg_active_counties', 'mean'),
        avg_county_turnover=('avg_county_turnover', 'mean'),
        avg_monthly_count=('avg_monthly_count', 'mean')
    )
    .sort_values(['avg_county_turnover', 'counties_visited'], ascending=False)
)

cluster_example_species = (
    flight_pattern_features
    .assign(cluster=flight_labels)
    .sort_values(['cluster', 'active_months', 'counties_visited', 'avg_monthly_count'], ascending=[True, False, False, False])
    .groupby('cluster')['common_name']
    .apply(lambda s: list(s.head(5)))
    .to_dict()
)

### Interpreting Clusters In Terms Of Movement Behavior

The analysis below combines the clustering output with county-level corridor inference so each cluster can be interpreted as a movement type.  
We map counties to their respective centroids, then find the coordinates for monthly cluster centers which are computed as weighted averages of the centroids of the cluster's counties, where larger weights are assigned to counties with more bird observations. Then we compare the monthly cluster centers across time to identify the clusters' movement paths and overall directions.

In [83]:
raw_ebird_df = pd.read_csv('../data/ebird_co.csv')

county_centroids = (
    raw_ebird_df
    .dropna(subset=['subnational2Name', 'lat', 'lng'])
    .groupby('subnational2Name', as_index=False)
    .agg(
        county_lat=('lat', 'mean'),
        county_lng=('lng', 'mean')
    )
    .rename(columns={'subnational2Name': 'county'})
)

flight_cluster_assignments = flight_pattern_features[['common_name']].copy()
flight_cluster_assignments['cluster'] = flight_labels

cluster_monthly_county_paths = (
    species_monthly_county
    .merge(flight_cluster_assignments, on='common_name', how='inner')
    .merge(county_centroids, on='county', how='left')
    .dropna(subset=['county_lat', 'county_lng'])
    .groupby(['cluster', 'month', 'county', 'county_lat', 'county_lng'], as_index=False)
    .agg(cluster_monthly_count=('monthly_count', 'sum'))
)

cluster_monthly_centers = (
    cluster_monthly_county_paths
    .groupby(['cluster', 'month'], as_index=False)
    .apply(
        lambda df: pd.Series({
            'weighted_lat': np.average(df['county_lat'], weights=df['cluster_monthly_count']),
            'weighted_lng': np.average(df['county_lng'], weights=df['cluster_monthly_count']),
            'total_cluster_count': df['cluster_monthly_count'].sum(),
            'top_counties': ' -> '.join(
                df.sort_values('cluster_monthly_count', ascending=False)['county'].head(3)
            )
        })
        , include_groups=False
    )
    .reset_index(drop=True)
    .sort_values(['cluster', 'month'])
)

def describe_direction(lat_change, lng_change, threshold=0.15):
    north_south = ''
    east_west = ''

    if lat_change > threshold:
        north_south = 'north'
    elif lat_change < -threshold:
        north_south = 'south'

    if lng_change > threshold:
        east_west = 'east'
    elif lng_change < -threshold:
        east_west = 'west'

    if north_south and east_west:
        return f'{north_south}{east_west}'
    if north_south:
        return north_south
    if east_west:
        return east_west
    return 'stable'

movement_rows = []
for cluster_id, cluster_df in cluster_monthly_centers.groupby('cluster'):
    cluster_df = cluster_df.sort_values('month').reset_index(drop=True)
    for i in range(len(cluster_df) - 1):
        start_row = cluster_df.iloc[i]
        end_row = cluster_df.iloc[i + 1]
        movement_rows.append({
            'cluster': cluster_id,
            'start_month': int(start_row['month']),
            'end_month': int(end_row['month']),
            'start_counties': start_row['top_counties'],
            'end_counties': end_row['top_counties'],
            'lat_change': end_row['weighted_lat'] - start_row['weighted_lat'],
            'lng_change': end_row['weighted_lng'] - start_row['weighted_lng'],
            'direction': describe_direction(
                end_row['weighted_lat'] - start_row['weighted_lat'],
                end_row['weighted_lng'] - start_row['weighted_lng']
            )
        })

cluster_direction_steps = pd.DataFrame(movement_rows)

cluster_path_summary = (
    cluster_direction_steps
    .groupby('cluster', as_index=False)
    .agg(
        first_active_month=('start_month', 'min'),
        last_active_month=('end_month', 'max'),
        representative_path=('start_counties', lambda s: ' | '.join(pd.Series(s).drop_duplicates().head(4))),
        dominant_directions=('direction', lambda s: ' -> '.join(pd.Series(s).replace('stable', np.nan).dropna().head(6)))
    )
)

overall_direction = (
    cluster_direction_steps
    .groupby('cluster', as_index=False)
    .agg(total_lat_change=('lat_change', 'sum'), total_lng_change=('lng_change', 'sum'))
)
overall_direction['overall_direction'] = overall_direction.apply(
    lambda row: describe_direction(row['total_lat_change'], row['total_lng_change'], threshold=0.3),
    axis=1
)

cluster_path_summary = cluster_path_summary.merge(overall_direction[['cluster', 'overall_direction']], on='cluster', how='left')
cluster_path_summary['dominant_directions'] = cluster_path_summary['dominant_directions'].replace('', 'stable')

month_names = {
    1: 'January', 2: 'February', 3: 'March', 4: 'April', 5: 'May', 6: 'June',
    7: 'July', 8: 'August', 9: 'September', 10: 'October', 11: 'November', 12: 'December'
}

# Function to convert numeric cluster metrics into plain-language traits
def classify_movement_traits(row):
    traits = []
    if row['avg_county_turnover'] >= 0.65:
        traits.append('high county turnover')
    elif row['avg_county_turnover'] >= 0.35:
        traits.append('moderate county turnover')
    else:
        traits.append('low county turnover')

    if row['active_months'] >= 9:
        traits.append('year-round or near year-round activity')
    elif row['active_months'] >= 5:
        traits.append('clear seasonal presence')
    else:
        traits.append('short or sporadic seasonal windows')

    if row['counties_visited'] >= 30:
        traits.append('broad statewide footprint')
    elif row['counties_visited'] >= 8:
        traits.append('regional multi-county footprint')
    else:
        traits.append('narrow county footprint')

    return ', '.join(traits)

# Use the path summary as a lookup table when writing the narrative for each cluster.
cluster_path_lookup = cluster_path_summary.set_index('cluster')
cluster_lines = ['### Cluster Interpretations']

# Render the final writeup
for _, summary_row in flight_cluster_summary.reset_index().sort_values('cluster').iterrows():
    cluster_num = int(summary_row['cluster'])
    path_row = cluster_path_lookup.loc[cluster_num]
    # Pull representative species examples and convert the metrics into plain-language traits.
    examples = ', '.join(f'**{species}**' for species in cluster_example_species.get(cluster_num, []))
    movement_traits = classify_movement_traits(summary_row)
    month_span = f"{month_names[int(path_row['first_active_month'])]} to {month_names[int(path_row['last_active_month'])]}"

    # Build one short paragraph block per cluster for the markdown report.
    cluster_lines.append(f"#### Cluster {cluster_num}")
    cluster_lines.append(
        f"This cluster contains **{int(summary_row['species_count'])} species** and is characterized by {movement_traits}. "
        f"On average, species in this cluster are active across **{summary_row['active_months']:.1f} months**, visit **{summary_row['counties_visited']:.1f} counties**, and show **{summary_row['avg_county_turnover']:.2f}** month-to-month county turnover."
    )
    cluster_lines.append(
        f"Across {month_span}, the representative county path is **{path_row['representative_path']}**. "
        f"The dominant month-to-month movement directions are **{path_row['dominant_directions']}**, and the overall drift is **{path_row['overall_direction']}**."
    )
    cluster_lines.append(f"Example species in this cluster include {examples}.")
    cluster_lines.append('')

display(Markdown('\n'.join(cluster_lines)))


### Cluster Interpretations
#### Cluster 0
This cluster contains **200 species** and is characterized by high county turnover, clear seasonal presence, regional multi-county footprint. On average, species in this cluster are active across **6.1 months**, visit **11.5 counties**, and show **0.75** month-to-month county turnover.
Across January to December, the representative county path is **Pueblo -> Jefferson -> Larimer | Pueblo -> Denver -> Larimer | Jefferson -> Larimer -> Denver | Yuma -> Washington -> Montezuma**. The dominant month-to-month movement directions are **east -> southeast -> southwest -> west -> east -> northeast**, and the overall drift is **stable**.
Example species in this cluster include **Boreal Owl**, **Red-necked Grebe**, **Pacific Loon**, **White-winged Crossbill**, **Helmeted Guineafowl**.

#### Cluster 1
This cluster contains **285 species** and is characterized by moderate county turnover, year-round or near year-round activity, broad statewide footprint. On average, species in this cluster are active across **10.7 months**, visit **43.6 counties**, and show **0.52** month-to-month county turnover.
Across January to December, the representative county path is **Larimer -> Jefferson -> Boulder | Larimer -> Boulder -> Jefferson | Larimer -> Boulder -> Lincoln | Boulder -> Larimer -> Arapahoe**. The dominant month-to-month movement directions are **northeast -> west -> east -> west**, and the overall drift is **stable**.
Example species in this cluster include **Northern Harrier**, **Prairie Falcon**, **Sharp-shinned Hawk**, **Golden Eagle**, **American Pipit**.

#### Cluster 2
This cluster contains **76 species** and is characterized by low county turnover, short or sporadic seasonal windows, narrow county footprint. On average, species in this cluster are active across **1.7 months**, visit **1.2 counties**, and show **0.03** month-to-month county turnover.
Across January to December, the representative county path is **Montezuma -> Arapahoe -> Larimer | Larimer -> Montezuma -> Arapahoe | Montezuma -> Larimer -> Baca | El Paso -> Larimer -> Montezuma**. The dominant month-to-month movement directions are **northeast -> southeast -> northeast -> east -> northwest -> southwest**, and the overall drift is **east**.
Example species in this cluster include **American Wigeon x Mallard (hybrid)**, **Steller's Jay x Woodhouse's Scrub-Jay (hybrid)**, **Canyon x Spotted Towhee (hybrid)**, **Crissal Thrasher**, **Bufflehead x Common Goldeneye (hybrid)**.

#### Cluster 3
This cluster contains **6 species** and is characterized by moderate county turnover, year-round or near year-round activity, broad statewide footprint. On average, species in this cluster are active across **11.5 months**, visit **46.0 counties**, and show **0.47** month-to-month county turnover.
Across January to December, the representative county path is **Kit Carson -> Baca -> Weld | Bent -> Boulder -> Mesa | Rio Grande -> Conejos -> Denver | Arapahoe -> Larimer -> Weld**. The dominant month-to-month movement directions are **southwest -> southwest -> northeast -> west -> east -> north**, and the overall drift is **northwest**.
Example species in this cluster include **Snow Goose**, **Sandhill Crane**, **American Crow**, **Ring-billed Gull**, **Cackling Goose**.


## Model Evaluation

In [84]:
from IPython.display import Markdown, display

evaluation_lines = [
    '### Evaluation Results',
    '',
    f"- **Species prevalence over time:** The selected cluster count was **{prevalence_best_k}**. The **Silhouette Score** was **{prevalence_metrics['silhouette_score']:.4f}**, and the **Davies-Bouldin Index** was **{prevalence_metrics['davies_bouldin_index']:.4f}**.",
    f"- **Flight-pattern types:** The selected cluster count was **{flight_best_k}**. The **Silhouette Score** was **{flight_metrics['silhouette_score']:.4f}**, and the **Davies-Bouldin Index** was **{flight_metrics['davies_bouldin_index']:.4f}**.",
    '',
    'Higher Silhouette Scores indicate better-separated clusters, while lower Davies-Bouldin values indicate tighter and more distinct cluster structure.'
]

display(Markdown('\n'.join(evaluation_lines)))

### Evaluation Results

- **Species prevalence over time:** The selected cluster count was **2**. The **Silhouette Score** was **0.4960**, and the **Davies-Bouldin Index** was **0.7972**.
- **Flight-pattern types:** The selected cluster count was **4**. The **Silhouette Score** was **0.5054**, and the **Davies-Bouldin Index** was **0.6172**.

Higher Silhouette Scores indicate better-separated clusters, while lower Davies-Bouldin values indicate tighter and more distinct cluster structure.

## Challenges and Solutions

- **Challenge:** Bird counts span very different scales across counties and species.  
  **Solution:** Standardization and `log1p` transforms were used to keep large counts from dominating Euclidean distance.

- **Challenge:** Flight paths are not directly observed, and the cleaned modeling table is aggregated at the county-month level rather than as individual tracked trajectories.  
  **Solution:** County centroids were estimated from the raw eBird latitude and longitude observations in `ebird_co.csv`, then monthly cluster centers were computed from county-level bird counts to infer likely corridors and overall movement direction.

- **Challenge:** Choosing `k` is subjective in unsupervised learning.  
  **Solution:** Multiple values of `k` were compared using both Silhouette Score and Davies-Bouldin Index.

## Final Interpretation Notes

- Higher Silhouette Score is better because it indicates tighter, better-separated clusters.
- Lower Davies-Bouldin Index is better because it indicates lower within-cluster dispersion relative to between-cluster separation.
- If either metric is weak for a task, the clusters may still be useful for exploration, but they should be interpreted as soft groupings rather than definitive ecological classes.